# Tiny Dreamer Highway — Colab H100 Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

Use this notebook only after the baseline real run is stable and you want to make better use of an H100 runtime.

## H100 intent

This notebook uses the larger H100-oriented config. Compare its runtime and final metrics against the baseline notebook, not against the sanity notebook.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet

Already up to date.


From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD


In [3]:
import json
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'h100_experiment.yaml'
config = load_experiment_config(CONFIG_PATH)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('Loaded config from:', CONFIG_PATH)
print('GPU:', gpu_name)
if 'H100' not in gpu_name:
    print('Warning: this notebook is intended for H100-class runtimes.')
print('Batch size:', config.training.batch_size)
print('World-model updates/cycle:', config.training.world_model_updates_per_cycle)
print('Behavior updates/cycle:', config.training.behavior_updates_per_cycle)

Loaded config from: /content/CSC_580_Final_Project/examples/h100_experiment.yaml
GPU: NVIDIA H100 80GB HBM3
Batch size: 128
World-model updates/cycle: 8
Behavior updates/cycle: 8


In [4]:
RUN_NAME = 'h100_real_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CYCLES = None
WARM_START_STEPS = None
POLICY_STEPS = None
CHECKPOINT_INTERVAL = None
RESUME_FROM = None

print('Run name:', RUN_NAME)
print('Effective cycles:', config.training.cycles if CYCLES is None else CYCLES)

Run name: h100_real_run_001
Effective cycles: 500


In [ ]:
print('Launching H100 run. Per-cycle progress lines will appear below.')

training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('Completed cycles:', training_summary.completed_cycles)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Latest metrics:', training_summary.latest_record)

Launching H100 run. Per-cycle progress lines will appear below.
[train] starting run | cycles=500 | start_step=1 | warm_start_steps=2000 | policy_steps=64 | device=cuda
[train] step=1/500 | warm=2000 | policy=64 | replay=2064 | world_total=3.2360 | actor=-0.8101 | critic=0.6885 | cycle_s=470.0 | elapsed_s=470.0 | checkpoint=-
[train] step=2/500 | warm=0 | policy=64 | replay=2128 | world_total=3.1504 | actor=-0.9970 | critic=0.9332 | cycle_s=15.0 | elapsed_s=485.0 | checkpoint=-
[train] step=3/500 | warm=0 | policy=64 | replay=2192 | world_total=3.0980 | actor=-0.9661 | critic=0.7817 | cycle_s=14.8 | elapsed_s=499.7 | checkpoint=-
[train] step=4/500 | warm=0 | policy=64 | replay=2256 | world_total=3.0847 | actor=-1.1652 | critic=1.0966 | cycle_s=14.8 | elapsed_s=514.5 | checkpoint=-
[train] step=5/500 | warm=0 | policy=64 | replay=2320 | world_total=3.0741 | actor=-1.2609 | critic=1.2116 | cycle_s=14.9 | elapsed_s=529.3 | checkpoint=-
[train] step=6/500 | warm=0 | policy=64 | replay=238

In [ ]:
summary_path = training_summary.log_dir / 'latest_summary.json'
summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
summary_payload

In [ ]:
from IPython.display import Image, display
from tiny_dreamer_highway.evaluation import export_training_history_artifacts

analysis_outputs = export_training_history_artifacts(
    training_summary.log_dir / 'cycle_metrics.csv',
    RUN_ARTIFACT_ROOT / 'analysis',
    prefix=RUN_NAME,
)

display(Image(filename=str(analysis_outputs['curves'])))
json.loads(analysis_outputs['summary'].read_text(encoding='utf-8'))

## Agent Driving Demo

Record a short GIF showing the trained policy driving in the real highway-env. The checkpoint from the run above is used to load the actor and world model, then the agent is rolled out for a few episodes.

In [ ]:
from IPython.display import Image, display
import importlib
import tiny_dreamer_highway.evaluation as evaluation_pkg

try:
    evaluation_pkg = importlib.reload(evaluation_pkg)
    record_demo_videos = evaluation_pkg.record_demo_videos
except (AttributeError, ImportError):
    from tiny_dreamer_highway.evaluation.policy_rollout import record_demo_videos

print('Using demo recorder from:', record_demo_videos.__module__)

demo_outputs = record_demo_videos(
    config,
    checkpoint_path=training_summary.latest_checkpoint,
    output_dir=RUN_ARTIFACT_ROOT / 'demo_videos',
    num_episodes=2,
    max_steps=100,
    fps=15,
    seed=config.seed,
    prefix=RUN_NAME,
    device=config.device,
)

for gif_path in demo_outputs.video_paths:
    print(f'\n{gif_path.name}')
    display(Image(filename=str(gif_path)))